This notebook walks through several ways to estimate uncertainties from a finite set of samples. We will compare several methods and see how close they are to the true value.



**Overview:** In this notebook you will
- Write a function that implements jackknife resampling
- Estimate the mean, $\mu$, and standard deviation, $\sigma$, of a set of mystery data using
  - separate sample sets
  - bootstrap resampling
  - jackknife resampling
- Compare the different methods. How close are the methods to each other? To the true value?

Look for `#! YOUR CODE HERE` comments for where to add your own solution

In [ ]:
# Import libraries
import numpy as np
from numpy.random import default_rng
from scipy.stats import bootstrap

import matplotlib.pyplot as plt

In [ ]:
# Function to calculate test statistic; mean by default
def myTestStatistic(X):
  """
  Calculates a test statistic on data sample X; shape=(n,)
  """
  return np.mean(X)

# Mystery data

In [ ]:
np.random.seed(7)
data = np.load('mysteryData.npy')
print("Number of data samples = ", data.shape[0])

# Estimate $\mu, \sigma^2$ via separate sample sets

In [ ]:
N = 5 # Number of test sets

# Split data into N test sets; since all data is iid, we don't need to randomize further
new_data = data.reshape(N, -1) # (N, 500//N)

Mhat = np.ones(N)*-999 # Initialize array with dummy values

for j in range(N):
  X = new_data[j,:]
  Mhat[j] = myTestStatistic(X)


separate_samples_mean = np.mean(Mhat)
separate_samples_std  = np.std(Mhat)

print("Values: ", Mhat)
print(f"Mean: {separate_samples_mean:0.4f}")
print(f"Std:  {separate_samples_std:0.4f}")

# Estimate $\mu, \sigma^2$ via bootstrap resampling

In [ ]:
nBootstrap  = 1000
method      = 'basic'

bootstrap_rng = default_rng(7)
test_rng = default_rng(42)

new_data = (data,) # Must be in a sequence
res = bootstrap(new_data, myTestStatistic, method=method, random_state=bootstrap_rng, n_resamples=nBootstrap)

# Version 1
bootstrap_mean = myTestStatistic(data)
bootstrap_std  = res.standard_error

# Version 2
avg_bootstrap_distb = np.mean(res.bootstrap_distribution)
std_bootstrap_distb = np.std(res.bootstrap_distribution, ddof=1) # https://github.com/scipy/scipy/issues/15160

print("Version 1:")
print(f"Mean: {bootstrap_mean:0.4f}") # This is the mean taken over the original sample
print(f"Std:  {bootstrap_std:0.4f}")
print("")
print("Version 2:")
print(f"Mean: {avg_bootstrap_distb:0.4f}") # This is the mean taken over the bootstrap distribution
print(f"Std:  {std_bootstrap_distb:0.4f}")

# Estimate $\mu, \sigma^2$ via jackknife resampling

In [ ]:
def jackknife_statistic(data, statistic):
  """
  data: ndarray of shape (n,)
  statistic: Callable function which takes in a data sample of size m and calculates the statistic on that sample
  """
  n = len(data)
  jackknife_samples = np.zeros(n)

  for i in range(n):
      sample = np.delete(data, i)
      jackknife_samples[i] = statistic(sample)

  #! YOUR CODE HERE
  # jackknife_mean     = #!
  # jackknife_variance = #!
  # jackknife_bias     = #!

  return jackknife_mean, jackknife_variance, jackknife_bias

In [ ]:
jackknife_mean, jackknife_variance, jackknife_bias = jackknife_statistic(data, myTestStatistic)

print(f"Mean: {jackknife_mean:0.4f}")
print(f"Std:  {jackknife_variance:0.4f}")
print(f"Bias: {jackknife_bias:0.4f}")

# Compare results

In [ ]:
#TRUE_MEAN, TRUE_STD =  #! YOUR CODE HERE (put in values from slide)
TRUE_STD_ERR = TRUE_STD / np.sqrt(len(data))

plt.errorbar(x=TRUE_MEAN, y=1, xerr=TRUE_STD_ERR, yerr=0, fmt='o', color='black')
plt.errorbar(x=separate_samples_mean, y=1.1, xerr=separate_samples_std, yerr=0, fmt='o', color='tab:blue')
plt.errorbar(x=bootstrap_mean, y=1.2, xerr=bootstrap_std, yerr=0, fmt='o', color='tab:orange')
plt.errorbar(x=avg_bootstrap_distb, y=1.3, xerr=std_bootstrap_distb, yerr=0, fmt='o', color='tab:red')
plt.errorbar(x=jackknife_mean, y=1.4, xerr=jackknife_variance, yerr=0, fmt='o', color='tab:purple')

plt.xlabel(r'$\mu$', fontsize=16)
plt.yticks([1, 1.1, 1.2, 1.3, 1.4], ['Truth', 'Separate Samples', 'Bootstrap (default)', 'Bootstrap (distb.)', 'Jackknife'], fontsize=14)
plt.show()

# Answers

Enter into the following markdown cells and uncomment to see the answers

`jackknife_statistic(data, statistic)`

<!-- ```
# https://www.geeksforgeeks.org/data-science/jackknife-resampling/
def jackknife_statistic(data, statistic):
  """
  data: ndarray of shape (n,)
  statistic: Callable function which takes in a data sample of size m and calculates the statistic on that sample
  """
  n = len(data)
  jackknife_samples = np.zeros(n)

  for i in range(n):
      sample = np.delete(data, i)
      jackknife_samples[i] = statistic(sample)

  jackknife_mean     = np.mean(jackknife_samples)
  jackknife_variance = (n - 1) / n * np.sum((jackknife_samples - jackknife_mean)**2)
  jackknife_bias     = (n - 1) * (jackknife_mean - statistic(data))
  
  return jackknife_mean, jackknife_variance, jackknife_bias
``` -->

`TRUE_MEAN, TRUE_STD =`

<!-- ```
TRUE_MEAN, TRUE_STD = 1.4, 2


Data was generated via:
```
np.random.seed(7)
data = np.random.normal(loc = 1.4, scale = 2, size = 500)
```
``` -->